In [1]:
# Notebook: nb_silver_cipc_clean
# Purpose: Clean, standardize, and deduplicate CIPC Bronze data

from pyspark.sql import functions as F
from datetime import datetime

print(" Starting Silver Layer CIPC Cleaning...")
print("=" * 60)

# 1. Read from Bronze
bronze_df = spark.table("bronze_cipc_companies")
print(f" Read {bronze_df.count()} rows from Bronze.")


StatementMeta(, 1a38fa07-484d-4f47-896b-d7a3b778aba4, 3, Finished, Available, Finished, False)

 Starting Silver Layer CIPC Cleaning...
 Read 2000 rows from Bronze.


In [2]:
# 2. Standardize Strings (Trim whitespace, Uppercase names)
clean_df = bronze_df.withColumn("CompanyName", F.upper(F.trim(F.col("CompanyName")))) \
                    .withColumn("Province", F.upper(F.trim(F.col("Province")))) \
                    .withColumn("CompanyType", F.upper(F.trim(F.col("CompanyType")))) \
                    .withColumn("Status", F.upper(F.trim(F.col("Status"))))

# 3. Standardize Registration Number (Remove spaces/dashes if any exist)
clean_df = clean_df.withColumn("RegistrationNumber", F.regexp_replace(F.col("RegistrationNumber"), "[\\s-]", ""))

# 4. Drop Exact Duplicates
initial_count = clean_df.count()
clean_df = clean_df.dropDuplicates()
print(f"🗑️ Removed {initial_count - clean_df.count()} exact duplicate rows.")

StatementMeta(, 1a38fa07-484d-4f47-896b-d7a3b778aba4, 4, Finished, Available, Finished, False)

🗑️ Removed 1000 exact duplicate rows.


In [3]:
# 5. Filter out "Deregistered" companies into an Archive table
# (This keeps the main Silver table clean for active business logic)
archive_df = clean_df.filter(F.col("Status") == "DEREGISTERED")
silver_active_df = clean_df.filter(F.col("Status") != "DEREGISTERED")

print(f" Archived {archive_df.count()} deregistered companies.")
print(f"✅ Kept {silver_active_df.count()} active companies for Silver layer.")


StatementMeta(, 1a38fa07-484d-4f47-896b-d7a3b778aba4, 5, Finished, Available, Finished, False)

 Archived 109 deregistered companies.
✅ Kept 891 active companies for Silver layer.


In [4]:
# 6. Add Data Quality & Load Timestamp Columns (Crucial for Silver!)
silver_active_df = silver_active_df.withColumn("dq_score", F.lit(100)) \
                                   .withColumn("silver_load_timestamp", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

archive_df = archive_df.withColumn("dq_score", F.lit(100)) \
                       .withColumn("silver_load_timestamp", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

# 7. Write to Silver Delta Tables
silver_active_df.write.format("delta").mode("overwrite").saveAsTable("silver_companies")
archive_df.write.format("delta").mode("overwrite").saveAsTable("silver_companies_archive")

print("\n Silver CIPC Cleaning Complete!")
print("Tables created: silver_companies, silver_companies_archive")

StatementMeta(, 1a38fa07-484d-4f47-896b-d7a3b778aba4, 6, Finished, Available, Finished, False)


 Silver CIPC Cleaning Complete!
Tables created: silver_companies, silver_companies_archive
